# Credit Score Prediction - Machine Learning Project
## Complete End-to-End Implementation

This notebook demonstrates credit score prediction using machine learning algorithms.

**Dataset:** Credit Score Classification from Kaggle

**Link:** https://www.kaggle.com/datasets/parisrohan/credit-score-classification

## 1. Import Libraries

In [ ]:
# Data manipulation
import pandas as pd
import numpy as np

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Machine Learning
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

# Settings
import warnings
warnings.filterwarnings('ignore')

# Set visualization style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

print("All libraries imported successfully!")

## 2. Load Dataset

In [ ]:
# Load the data
df = pd.read_csv('train.csv')

# Display basic information
print(f"Dataset Shape: {df.shape}")
print(f"\nNumber of rows: {df.shape[0]}")
print(f"Number of columns: {df.shape[1]}")

# Display first few rows
df.head()

## 3. Exploratory Data Analysis (EDA)

In [ ]:
# Dataset information
print("Dataset Information:")
print("="*50)
df.info()

In [ ]:
# Statistical summary
df.describe()

In [ ]:
# Check for missing values
missing_values = df.isnull().sum()
missing_percent = (missing_values / len(df)) * 100

missing_df = pd.DataFrame({
    'Missing_Count': missing_values,
    'Percentage': missing_percent
})

missing_df = missing_df[missing_df['Missing_Count'] > 0].sort_values('Missing_Count', ascending=False)

if len(missing_df) > 0:
    print("Missing Values:")
    print(missing_df)
else:
    print("No missing values found!")

In [ ]:
# Target variable distribution
if 'Credit_Score' in df.columns:
    print("Credit Score Distribution:")
    print(df['Credit_Score'].value_counts())
    print(f"\nPercentage Distribution:")
    print(df['Credit_Score'].value_counts(normalize=True) * 100)
    
    # Visualize
    plt.figure(figsize=(10, 6))
    df['Credit_Score'].value_counts().plot(kind='bar', color=['#2ecc71', '#f39c12', '#e74c3c'])
    plt.title('Credit Score Distribution', fontsize=16, fontweight='bold')
    plt.xlabel('Credit Score Category', fontsize=12)
    plt.ylabel('Count', fontsize=12)
    plt.xticks(rotation=45)
    plt.grid(axis='y', alpha=0.3)
    plt.tight_layout()
    plt.show()

In [ ]:
# Correlation heatmap for numerical features
numerical_cols = df.select_dtypes(include=[np.number]).columns

if len(numerical_cols) > 0:
    plt.figure(figsize=(14, 10))
    correlation_matrix = df[numerical_cols].corr()
    sns.heatmap(correlation_matrix, annot=False, cmap='coolwarm', center=0)
    plt.title('Correlation Heatmap of Numerical Features', fontsize=16, fontweight='bold')
    plt.tight_layout()
    plt.show()

## 4. Data Preprocessing

In [ ]:
# Create a copy for preprocessing
df_processed = df.copy()

# Remove ID columns if present
cols_to_drop = ['ID', 'Customer_ID', 'Name', 'SSN']
cols_to_drop = [col for col in cols_to_drop if col in df_processed.columns]

if cols_to_drop:
    df_processed.drop(cols_to_drop, axis=1, inplace=True)
    print(f"Dropped columns: {cols_to_drop}")

print(f"\nProcessed dataset shape: {df_processed.shape}")

In [ ]:
# Handle missing values
print("Handling missing values...")

for col in df_processed.columns:
    if df_processed[col].isnull().sum() > 0:
        if df_processed[col].dtype == 'object':
            # For categorical: use mode
            df_processed[col].fillna(df_processed[col].mode()[0], inplace=True)
            print(f"  Filled {col} (categorical) with mode")
        else:
            # For numerical: use median
            df_processed[col].fillna(df_processed[col].median(), inplace=True)
            print(f"  Filled {col} (numerical) with median")

print("\nMissing values after handling:")
print(df_processed.isnull().sum().sum())

In [ ]:
# Encode categorical variables
label_encoders = {}
categorical_cols = df_processed.select_dtypes(include=['object']).columns.tolist()

# Remove target from encoding list
if 'Credit_Score' in categorical_cols:
    categorical_cols.remove('Credit_Score')

print(f"Encoding {len(categorical_cols)} categorical columns...")

for col in categorical_cols:
    le = LabelEncoder()
    df_processed[col] = le.fit_transform(df_processed[col].astype(str))
    label_encoders[col] = le
    print(f"  Encoded: {col}")

# Encode target variable
if 'Credit_Score' in df_processed.columns:
    le_target = LabelEncoder()
    df_processed['Credit_Score'] = le_target.fit_transform(df_processed['Credit_Score'])
    label_encoders['Credit_Score'] = le_target
    print(f"\nTarget variable encoded: Credit_Score")
    print(f"Classes: {le_target.classes_}")

## 5. Feature Preparation

In [ ]:
# Separate features and target
X = df_processed.drop('Credit_Score', axis=1)
y = df_processed['Credit_Score']

print(f"Features shape: {X.shape}")
print(f"Target shape: {y.shape}")
print(f"\nFeature columns: {list(X.columns)}")

In [ ]:
# Split the data
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Training set size: {X_train.shape[0]}")
print(f"Test set size: {X_test.shape[0]}")
print(f"\nClass distribution in training set:")
print(y_train.value_counts())
print(f"\nClass distribution in test set:")
print(y_test.value_counts())

In [ ]:
# Scale the features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Features scaled successfully!")
print(f"\nScaled training data shape: {X_train_scaled.shape}")
print(f"Scaled test data shape: {X_test_scaled.shape}")

## 6. Model Training

In [ ]:
# Initialize models
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42),
    'Gradient Boosting': GradientBoostingClassifier(n_estimators=100, random_state=42)
}

print("Models initialized:")
for name in models.keys():
    print(f"  - {name}")

In [ ]:
# Train and evaluate models
results = {}

for name, model in models.items():
    print(f"\n{'='*50}")
    print(f"Training {name}...")
    print(f"{'='*50}")
    
    # Train the model
    model.fit(X_train_scaled, y_train)
    
    # Make predictions
    y_pred = model.predict(X_test_scaled)
    
    # Calculate accuracy
    accuracy = accuracy_score(y_test, y_pred)
    results[name] = accuracy
    
    print(f"Accuracy: {accuracy:.4f}")
    print(f"Training completed!")

## 7. Model Evaluation

In [ ]:
# Compare model performances
results_df = pd.DataFrame(list(results.items()), columns=['Model', 'Accuracy'])
results_df = results_df.sort_values('Accuracy', ascending=False)

print("\nModel Performance Comparison:")
print("="*50)
print(results_df.to_string(index=False))

# Visualize
plt.figure(figsize=(10, 6))
plt.barh(results_df['Model'], results_df['Accuracy'], color=['#3498db', '#2ecc71', '#e74c3c'])
plt.xlabel('Accuracy', fontsize=12)
plt.title('Model Performance Comparison', fontsize=16, fontweight='bold')
plt.xlim(0, 1)
for i, v in enumerate(results_df['Accuracy']):
    plt.text(v + 0.01, i, f'{v:.4f}', va='center')
plt.tight_layout()
plt.show()

In [ ]:
# Detailed evaluation for each model
for name, model in models.items():
    print(f"\n{'='*60}")
    print(f"{name} - Detailed Evaluation")
    print(f"{'='*60}")
    
    # Predictions
    y_pred = model.predict(X_test_scaled)
    
    # Classification Report
    print("\nClassification Report:")
    target_names = label_encoders['Credit_Score'].classes_
    print(classification_report(y_test, y_pred, target_names=target_names))
    
    # Confusion Matrix
    cm = confusion_matrix(y_test, y_pred)
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
                xticklabels=target_names, yticklabels=target_names)
    plt.title(f'Confusion Matrix - {name}', fontsize=14, fontweight='bold')
    plt.xlabel('Predicted', fontsize=12)
    plt.ylabel('Actual', fontsize=12)
    plt.tight_layout()
    plt.show()

## 8. Feature Importance Analysis

In [ ]:
# Random Forest Feature Importance
rf_model = models['Random Forest']
feature_names = X.columns
importances = rf_model.feature_importances_
indices = np.argsort(importances)[::-1]

# Create dataframe
feature_importance_df = pd.DataFrame({
    'Feature': [feature_names[i] for i in indices],
    'Importance': importances[indices]
})

print("Top 15 Most Important Features:")
print("="*50)
print(feature_importance_df.head(15).to_string(index=False))

# Visualize
plt.figure(figsize=(12, 8))
top_n = min(15, len(feature_names))
plt.barh(range(top_n), importances[indices[:top_n]], color='teal')
plt.yticks(range(top_n), [feature_names[i] for i in indices[:top_n]])
plt.xlabel('Importance Score', fontsize=12)
plt.title('Top 15 Feature Importances (Random Forest)', fontsize=14, fontweight='bold')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

## 9. Making Predictions on New Data

In [ ]:
# Example: Predict for a single sample
# You need to create a sample with the same features as your dataset

def predict_credit_score(model, scaler, sample_data, label_encoder):
    """
    Predict credit score for new data
    
    Parameters:
    - model: trained model
    - scaler: fitted scaler
    - sample_data: list of feature values
    - label_encoder: encoder for target variable
    """
    # Scale the sample
    sample_scaled = scaler.transform([sample_data])
    
    # Predict
    prediction = model.predict(sample_scaled)
    prediction_proba = model.predict_proba(sample_scaled)
    
    # Decode prediction
    credit_score = label_encoder.inverse_transform(prediction)[0]
    
    return credit_score, prediction_proba[0]

# Example usage (you need to provide actual feature values)
print("\nExample Prediction Function Ready!")
print("To make a prediction, create a sample with all feature values and call:")
print("predict_credit_score(models['Random Forest'], scaler, sample_data, label_encoders['Credit_Score'])")

## 10. Summary & Conclusions

In [ ]:
print("\n" + "="*60)
print("PROJECT SUMMARY")
print("="*60)

print(f"\nDataset: Credit Score Classification")
print(f"Total Samples: {len(df)}")
print(f"Number of Features: {X.shape[1]}")
print(f"Target Classes: {label_encoders['Credit_Score'].classes_}")

print("\nModel Performance:")
print("-" * 40)
for model_name, accuracy in sorted(results.items(), key=lambda x: x[1], reverse=True):
    print(f"{model_name:25s}: {accuracy:.4f}")

best_model = max(results, key=results.get)
print(f"\nBest Model: {best_model}")
print(f"Best Accuracy: {results[best_model]:.4f}")

print("\n" + "="*60)
print("PROJECT COMPLETED SUCCESSFULLY!")
print("="*60)